### Tool 使用的概述

#### 1. 工具的调用方式

##### 1.1 方式1: 直接调用

In [1]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    '''
    获取指定城市的天气信息
    参数: 
        city: 城市名称, 如 "北京", "上海"
    返回:
        天气信息字符串
    '''
    # 你的实现
    return city + "晴天, 温度 15 ℃"

In [3]:
get_weather.invoke({
    "city": "北京"
})

'北京晴天, 温度 15 ℃'

#### 1.2 方式2: 基于模型进行调用

In [4]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from rich import print as rprint

load_dotenv(override=True)
ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_API_BASE_URL = os.getenv("ZHIPUAI_API_BASE_URL")

# 获取大模型
model = init_chat_model(
    model='glm-5.3',
    model_provider="openai",
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_API_BASE_URL
)

In [5]:
from langchain_core.tools import tool

# 定义工具
@tool
def get_weather(city: str) -> str:
    '''
    获取指定城市的天气信息
    参数: 
        city: 城市名称, 如 "北京", "上海"
    返回:
        天气信息字符串
    '''
    # 你的实现
    return city + "晴天, 温度 15 ℃"

# 绑定工具
model_with_tools = model.bind_tools([get_weather])

# AI 可以决定是否调用工具
response = model_with_tools.invoke("北京天气如何?")

# 检查 AI 是否要调用工具
if response.tool_calls:
    print("AI 想调用工具: ", response.tool_calls)
else:
    print("AI 直接回答: ", response.content)

AI 想调用工具:  [{'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'call_-7101058168519649999', 'type': 'tool_call'}]


#### 2. 从 Message 流转看工具的调用

In [ ]:
from langchain.messages import HumanMessage, ToolMessage
from rich import print as rprint

@tool
def get_weather(city: str):
    '''获取天气的工具'''
    return f"{city}天气晴朗"

# 将模型和工具绑定
model_with_tools = model.bind_tools([get_weather])

# 声明一个消息列表
messages = [
    HumanMessage("今天北京天气如何")
]

# 模型生成调用工具请求
response = model_with_tools.invoke(messages)

# 添加 AIMessage 到 消息列表中
messages.append(response)

rprint(response)

tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weather":
        # 大模型和 Agent 的主要区别在于: 大模型不会主动的调用工具, 所以这时候我们需要主动让工具调用
        # 返回的是 ToolMessage 类型消息, 添加到消息列表中
        tool_response = get_weather.invoke(tool_call)
        print(type(tool_response))
        messages.append(tool_response)

print(f"{'='*20}> messages <{'='*20}")
for msg in messages:
    msg.pretty_print()
print(f"{'='*20}> messages <{'='*20}")
final_response = model_with_tools.invoke(messages)
print(f"final_response: \n{final_response}")

AIMessage(
    content='',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 33,
            'prompt_tokens': 155,
            'total_tokens': 188,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 21,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'glm-5.3',
        'system_fingerprint': None,
        'id': '20261110152044be53b4e6830b416b',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--01a1c454-a479-7501-973e-9c0e68ec0057-0',
    tool_calls=[
        {'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'call_-7101047482641018247', 'type': 'tool_call'}
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 155,
        'output_tokens': 33,
        'total_tokens': 188,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 21}
    }
)

<class 'langchain_core.messages.tool.ToolMessage'>
====================> messages <====================
================================ Human Message =================================

今天北京天气如何
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_-7101047482641018247)
 Call ID: call_-7101047482641018247
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

北京天气晴朗
====================> messages <====================
final_response: 
content='今天北京的天气晴朗，是个不错的天气！☀️ 适合外出活动，不过出门也可以注意防晒哦。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 175, 'total_tokens': 223, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 22, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai'